# Lab 01 - Explore Microsoft Web IQ APIs

## Required permissions

A Web IQ profile API key authorized for Web Search, Browse, News, Images and Videos. No Azure RBAC or tenant administrator role is required to run this notebook.

**Goal:** retrieve fresh public evidence, inspect sources, and choose the right API for a grounding task. Allow 40 minutes.

Web IQ is a web intelligence service, not an LLM. Its Azure resource provides billing; API requests go to `https://api.microsoft.ai/v3`.

**Before running:** set `WEBIQ_API_KEY` in `.env` or enter it at the hidden prompt. See the root README for Python/kernel installation. Only use public, non-sensitive queries. Calls incur usage charges; avoid repeatedly running all cells.

**Outcomes:** explain passages versus full documents, distinguish crawl dates from publication dates, retain source attribution, handle crawl/retry responses, explore image and video results, and prepare an evidence-backed briefing.

## 1. Configuration and authentication

Settings come from the deployment's `.env.azure`, then your optional `.env`. Existing environment variables take precedence. A hidden prompt avoids placing a key in notebook source or outputs.

This lab uses a **Web IQ profile API key** in `x-apikey`, not an Azure resource key. Production Entra authentication requires an app-only token for `https://api.microsoft.ai/.default` and a client ID bound to the Web IQ profile; a developer's `az login` token is not a substitute.

In [1]:
print('hello')

hello


In [2]:
import getpass
import json
import os
import time
from pathlib import Path

import requests
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open this notebook from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')

api_key = os.getenv('WEBIQ_API_KEY') or getpass.getpass('Web IQ API key: ')
if not api_key.strip():
    raise ValueError('A Web IQ portal API key is required.')
print('Configuration loaded. The API key will not be displayed.')

Configuration loaded. The API key will not be displayed.


## 2. Search the web for grounding passages

The documented starting point is `maxResults=10`, `contentFormat='passage'`, `maxLength=3000`. A passage is query-relevant evidence, not a generated answer or a full webpage. Web Search does **not** return a `snippet` field.

Inspect titles, URLs, content and optional timestamps. A successful response may omit `webResults` when there are no matches. Never invent missing dates or sources.

In [3]:
question = 'What are the latest Microsoft announcements about AI agents?'
response = requests.post(
    'https://api.microsoft.ai/v3/search/web',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': question, 'maxResults': 10,
        'contentFormat': 'passage', 'maxLength': 3000,
        'language': 'en', 'region': 'US', 'safeSearch': 'strict',
    },
    timeout=(10, 90),
)
response.raise_for_status()
web = response.json()
print(json.dumps(web, indent=2, ensure_ascii=False))

{
  "webResults": [
    {
      "title": "Microsoft Build 2026: Building agentic apps with Microsoft Fabric and Microsoft Databases | Microsoft Azure Blog",
      "url": "https://azure.microsoft.com/en-us/blog/microsoft-build-2026-building-agentic-apps-with-microsoft-fabric-and-microsoft-databases/",
      "content": "Microsoft Build 2026: Building agentic apps with Microsoft Fabric and Microsoft Databases | Microsoft Azure Blog Blog home Microsoft Build 2026: Building agentic apps with Microsoft Fabric and Microsoft Databases June 2 14 min read Microsoft Build 2026: Building agentic apps with Microsoft Fabric and Microsoft Databases By Arun Ulag, Executive Vice President, Microsoft Azure Team Listen to this post 0:00 / 0:00 1x Categories AI + machine learning Databases Hybrid + multicloud Internet of things Security Microsoft Fabric Business decision makers Developers Announcements Azure Cosmos DB Azure HorizonDB Microsoft Defender for Cloud Microsoft Foundry Show more In this article

**Checkpoint:** which two sources would you trust for the question, and why? Does `crawledAt` mean publication date? Could the first-ranked source still be outdated?

## 3. Localization and deliberate source scoping

Compare the English/US search with Spanish/Spain. The language parameter controls interface language; it does not guarantee that every returned document is in that language. A `site:` operator narrows recall and should be used only when the task needs it. `safeSearch='strict'` is not an absolute content-safety guarantee, especially with `site:`.

In [5]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/web',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'novedades de agentes de inteligencia artificial de Microsoft site:learn.microsoft.com',
        'language': 'es', 'region': 'ES', 'safeSearch': 'strict',
        'maxResults': 10, 'contentFormat': 'passage', 'maxLength': 3000,
    },
    timeout=(10, 90),
)
response.raise_for_status()
localized = response.json()
print(json.dumps(localized, indent=2, ensure_ascii=False))

{
  "webResults": [
    {
      "title": "Novedades de Copilot Studio - Microsoft Copilot Studio | Microsoft Learn",
      "url": "https://learn.microsoft.com/es-es/microsoft-copilot-studio/whats-new",
      "content": "Novedades de Copilot Studio - Microsoft Copilot Studio | Microsoft Learn Learn Microsoft Copilot Studio Novedades de Copilot Studio En este artículo Versiones lanzadas Cambios destacados En este artículo se proporcionan recursos para obtener información sobre las nuevas características de Copilot Studio. Versiones lanzadas Obtén una visión general de las nuevas capacidades en la hoja de ruta. Puedes filtrar por productos y áreas específicas y ver qué está previsto y cuándo. Nota: Las versiones se implementan durante varios días. La nueva o actualizada funcionalidad puede no aparecer inmediatamente. Cambios destacados Las siguientes secciones enumeran las características publicadas en los últimos meses, con vínculos a información relacionada. Octubre de 2026 (Vista previ

## 4. Browse: from a discovered source to page content

In the previous sections, we used **Search**, which starts with a question and finds relevant pages (`POST /v3/search/web`). Now we will look at **Browse**, which starts with a URL and retrieves content from that specific page (`POST /v3/browse`). These are two separate API calls: Search finds pages; Browse reads a page you choose.

**`liveCrawl` controls whether Browse can fetch a public page directly from its website when the URL is not in Web IQ's index.**

- **`'none'` (default):** return indexed content only; an unindexed URL returns HTTP 404.
- **`'fallback'` (used below):** return indexed content if available; otherwise, request a live crawl. HTTP 202 means the crawl is still processing.

**When it helps:** your application may already know a public URL, such as a newly published announcement or a less-discoverable documentation page, that Web IQ has not indexed yet. Fallback crawling can retrieve that page for grounding instead of stopping at a missing-index result. It does **not** force a refresh of an already-indexed page or guarantee the latest content; inspect `crawledAt` to assess when the page was last crawled.

**Limits:** crawling depends on publisher restrictions and availability. Browse cannot sign in to a website or accept its login credentials or session cookies; the Web IQ API key authenticates only to Web IQ. For protected content, use an authorized API or connector, such as the SharePoint approaches in labs 07-08. Live crawling does not bypass login pages or paywalls.

The earlier Web Search cell stored its JSON response in `web`. Its `webResults` list contains the pages Search found; `web.get('webResults', [])` reads that list, or returns an empty list if there were no results. Below, we select the first result, put its URL in `browse_url`, and print it so you can see exactly which page Browse will read. You can also set `browse_url` to a public URL you already know, without using Search first. The request uses `contentFormat='passage'` and `query=question` to retrieve relevant excerpts from that page. Full-document formats (`markdown`, `text`, `html`) require additional profile permission and return HTTP 403 without it.

The loop makes up to four requests. For a pending crawl it waits for the returned `retryAfter` duration, such as `60s`. Missing delays, waits longer than two minutes, and unfinished crawls raise explicit errors instead of being treated as page content.

In [7]:
search_results = web.get('webResults', [])
if not search_results or not search_results[0].get('url'):
    raise RuntimeError('No source URL to browse. Change the search query and rerun search.')

first_result = search_results[0]
browse_url = first_result['url']
print('Reading page:', browse_url)

for attempt in range(4):
    response = requests.post(
        'https://api.microsoft.ai/v3/browse',
        headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
        json={
            'url': browse_url, 'query': question, 'contentFormat': 'passage',
            'maxLength': 3000, 'liveCrawl': 'fallback',
        },
        timeout=(10, 90),
    )
    response.raise_for_status()
    page = response.json()
    print(json.dumps(page, indent=2, ensure_ascii=False))
    if response.status_code == 200:
        break
    if response.status_code != 202:
        raise RuntimeError(f'Unexpected Browse status: {response.status_code}.')
    if attempt == 3:
        raise TimeoutError('Browse did not complete after four requests. Retry later.')
    if not page.get('retryAfter'):
        raise RuntimeError('Pending crawl omitted retryAfter.')
    seconds = float(page['retryAfter'].removesuffix('s'))
    if not 0 <= seconds <= 120:
        raise TimeoutError(f'Server requested a {seconds}s wait; retry later manually.')
    print(f'Crawl pending. Waiting {seconds:g}s...')
    time.sleep(seconds)

Reading page: https://azure.microsoft.com/en-us/blog/microsoft-build-2026-building-agentic-apps-with-microsoft-fabric-and-microsoft-databases/


HTTPError: 403 Client Error: Forbidden for url: https://api.microsoft.ai/v3/browse

## 5. News: publisher attribution and current events

News results can include `source`, `lastUpdatedAt`, `crawledAt` and a thumbnail, but these fields are optional. Compare the publishers and date coverage against Web Search. News Search has a maximum of 20 results; do not assume it supports the video endpoint's `freshness` filter.

In [ ]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/news',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': question, 'maxResults': 5, 'language': 'en', 'region': 'US',
        'contentFormat': 'passage', 'maxLength': 3000,
    },
    timeout=(10, 90),
)
response.raise_for_status()
news = response.json()
print(json.dumps(news, indent=2, ensure_ascii=False))

## 6. Image and video discovery

These are discovery results, not permission to reuse media. After inspecting the image response JSON, run the preview cell to display the returned thumbnails with their titles and source pages. Previews load from external URLs in your notebook viewer; they do not send your Web IQ API key to the image host. Videos remain metadata-only.

Image filters include aspect ratio and minimum size. Video filters include duration, minimum resolution, and discovery-date freshness. An absent summary or moments array simply means that metadata is unavailable.

In [ ]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/images',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'Microsoft datacenter renewable energy', 'maxResults': 3,
        'aspectRatio': 'wide', 'imageSize': 'large', 'safeSearch': 'strict',
    },
    timeout=(10, 90),
)
response.raise_for_status()
images = response.json()
print(json.dumps(images, indent=2, ensure_ascii=False))

In [ ]:
from IPython.display import Image, display

image_results = images.get('imageResults', [])
if not image_results:
    print('No images returned. Try another image search query.')

for image in image_results:
    print('Title:', image.get('title', '(unavailable)'))
    print('Source page:', image.get('hostPageUrl', '(unavailable)'))
    preview_url = image.get('thumbnailUrl') or image.get('url')
    if preview_url and preview_url.startswith(('https://', 'http://')):
        display(Image(url=preview_url, width=400, embed=False))
    else:
        print('No preview URL available.')

In [ ]:
response = requests.post(
    'https://api.microsoft.ai/v3/search/videos',
    headers={'x-apikey': api_key, 'Content-Type': 'application/json'},
    json={
        'query': 'Microsoft Foundry agents tutorial', 'maxResults': 3,
        'duration': 'medium', 'resolution': '720p',
        'freshness': 'year', 'safeSearch': 'strict',
    },
    timeout=(10, 90),
)
response.raise_for_status()
videos = response.json()
print(json.dumps(videos, indent=2, ensure_ascii=False))

## 7. Challenge: build an evidence-backed briefing

Choose a public topic relevant to your audience. Retrieve three sources and write a three-claim briefing with a URL supporting each claim. Identify a missing date or conflicting claim rather than filling it in. Use Browse to retrieve relevant excerpts from a selected source and check whether they change your interpretation of the Search result.

**Acceptance criteria:** three source-backed claims, no invented dates, one stated uncertainty, and an explanation of why you chose Web Search, News or Browse. Retain source attribution. No LLM is required for this exercise; lab 02 adds agent orchestration.

Change `question` and rerun only the relevant request cells. Do not paste confidential customer or internal project details into public search queries.

## Continue to lab 02

Open `02-foundry-web-iq-agent.ipynb` to let a Foundry agent call this same service through MCP. Notice the distinction: REST gives you explicit retrieval control; the agent adds model-driven tool selection, approvals and synthesis.

Before sharing this notebook, clear all outputs. Returned web content, trace IDs and URLs may still reveal the topics you searched for. Follow the root README cleanup steps after the workshop.

References: [REST quickstart](https://learn.microsoft.com/web-iq/get-started/quickstart), [Web Search](https://learn.microsoft.com/web-iq/reference/web-search), [Browse](https://learn.microsoft.com/web-iq/reference/browse), [News](https://learn.microsoft.com/web-iq/reference/news-search), [Images](https://learn.microsoft.com/web-iq/reference/image-search), [Videos](https://learn.microsoft.com/web-iq/reference/video-search).